# VGGT + Any4D + CoTracker + DROID-W Clean Evaluation Notebook

目标：

1. 保存当前 `DROID-W + Ours` 输出。
2. 重新跑原始 `DROID-W baseline`。
3. 可选重新跑 `DROID-W + Ours`。
4. 统一计算并对比 ATE / RPE / runtime / static-weight 统计。
5. 生成论文可用的表格和相机轨迹图。

> 重要：baseline 和 ours 必须保存到不同目录，否则后一次运行会覆盖前一次结果。

## 0. Mount Drive and define paths

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

REPO = Path("/content/vggt-demo")
DROIDW = REPO / "third_party" / "DROID-W"

# The original DROID-W output path used by both baseline and ours during one run.
DROIDW_SCENE_OUT = DROIDW / "Outputs" / "Bonn" / "bonn_person_tracking"

# Permanent comparison folders. These folders avoid overwrite.
COMPARE_ROOT = REPO / "outputs_compare"
BASELINE_OUT = COMPARE_ROOT / "droidw_baseline"
OURS_OUT = COMPARE_ROOT / "droidw_ours"
EVAL_OUT = COMPARE_ROOT / "eval"

# Optional Drive backup.
DRIVE_BACKUP_ROOT = Path("/content/drive/MyDrive/vggt_demo_results/bonn_person_tracking_compare")

print("REPO:", REPO)
print("DROIDW:", DROIDW)
print("DROIDW_SCENE_OUT:", DROIDW_SCENE_OUT)
print("BASELINE_OUT:", BASELINE_OUT)
print("OURS_OUT:", OURS_OUT)
print("EVAL_OUT:", EVAL_OUT)


## 1. Clone or update your GitHub repo

In [ ]:
%%bash
set -e

cd /content

if [ ! -d "/content/vggt-demo/.git" ]; then
  git clone https://github.com/ZexuZang/vggt-demo.git
fi

cd /content/vggt-demo
git status
git branch --show-current


## 2. Check DROID-W environment

This assumes DROID-W has already been installed and patched. It only checks imports and the required folders. Do not use this cell to reinstall the whole environment.

In [ ]:
%%bash
set -e

cd /content/vggt-demo/third_party/DROID-W

TORCH_LIB=$(python - <<'PY'
import torch, os
print(os.path.join(os.path.dirname(torch.__file__), "lib"))
PY
)

export LD_LIBRARY_PATH="$TORCH_LIB:$LD_LIBRARY_PATH"
export PYTHONPATH="/content/vggt-demo/third_party/DROID-W:$PYTHONPATH"

python - <<'PY'
import torch
print("torch:", torch.__version__)
try:
    import droid_backends
    print("droid_backends OK")
except Exception as e:
    print("droid_backends failed:", repr(e))

try:
    import lietorch
    print("lietorch OK")
except Exception as e:
    print("lietorch failed:", repr(e))

try:
    from simple_knn._C import distCUDA2
    print("simple_knn OK")
except Exception as e:
    print("simple_knn failed:", repr(e))

try:
    import diff_gaussian_rasterization
    print("diff_gaussian_rasterization OK")
except Exception as e:
    print("diff_gaussian_rasterization failed:", repr(e))
PY

echo "DROID-W output folder:"
ls -lah Outputs/Bonn || true


## 3. Verify baseline and ours config

In [ ]:
%cd /content/vggt-demo/third_party/DROID-W

print("==== Baseline config check ====")
!grep -n "ours_dynamic\|static_weight\|scene\|output\|dir" configs/Dynamic/Bonn/bonn_person_tracking.yaml || true

print("\n==== Ours config check ====")
!grep -n "ours_dynamic\|static_weight\|scene\|output\|dir" configs/Dynamic/Bonn/person_tracking_ours.yaml || true

print("\n==== Static weight files ====")
!find /content/vggt-demo/outputs/droidw_ours_weights/person_tracking/static_weight_npy -type f | head -10 || true
!find /content/vggt-demo/outputs/droidw_ours_weights/person_tracking/static_weight_npy -type f | wc -l || true


## 4. Save current DROID-W output as Ours

Run this immediately after you have run `person_tracking_ours.yaml`. It protects the current output from being overwritten by the baseline run.

In [ ]:
%%bash
set -e

cd /content/vggt-demo

mkdir -p /content/vggt-demo/outputs_compare
mkdir -p /content/drive/MyDrive/vggt_demo_results/bonn_person_tracking_compare

if [ ! -d "/content/vggt-demo/third_party/DROID-W/Outputs/Bonn/bonn_person_tracking" ]; then
  echo "ERROR: current DROID-W output not found."
  exit 1
fi

echo "==== Save current output as Ours ===="
rm -rf /content/vggt-demo/outputs_compare/droidw_ours
cp -a /content/vggt-demo/third_party/DROID-W/Outputs/Bonn/bonn_person_tracking \
      /content/vggt-demo/outputs_compare/droidw_ours

rm -rf /content/drive/MyDrive/vggt_demo_results/bonn_person_tracking_compare/droidw_ours
cp -a /content/vggt-demo/third_party/DROID-W/Outputs/Bonn/bonn_person_tracking \
      /content/drive/MyDrive/vggt_demo_results/bonn_person_tracking_compare/droidw_ours

echo "Saved Ours output:"
find /content/vggt-demo/outputs_compare/droidw_ours -maxdepth 3 -type f | grep -Ei "pose|traj|metric|timer|png|txt|csv" | head -50


## 5. Run original DROID-W baseline and save separately

In [ ]:
%%bash
set -e

cd /content/vggt-demo/third_party/DROID-W

TORCH_LIB=$(python - <<'PY'
import torch, os
print(os.path.join(os.path.dirname(torch.__file__), "lib"))
PY
)

export LD_LIBRARY_PATH="$TORCH_LIB:$LD_LIBRARY_PATH"
export PYTHONPATH="/content/vggt-demo/third_party/DROID-W:$PYTHONPATH"

echo "==== Remove shared output folder to avoid mixing results ===="
rm -rf Outputs/Bonn/bonn_person_tracking

echo "==== Run original DROID-W baseline ===="
python run.py --config configs/Dynamic/Bonn/bonn_person_tracking.yaml

echo "==== Save baseline output ===="
mkdir -p /content/vggt-demo/outputs_compare
rm -rf /content/vggt-demo/outputs_compare/droidw_baseline
cp -a Outputs/Bonn/bonn_person_tracking /content/vggt-demo/outputs_compare/droidw_baseline

mkdir -p /content/drive/MyDrive/vggt_demo_results/bonn_person_tracking_compare
rm -rf /content/drive/MyDrive/vggt_demo_results/bonn_person_tracking_compare/droidw_baseline
cp -a Outputs/Bonn/bonn_person_tracking \
      /content/drive/MyDrive/vggt_demo_results/bonn_person_tracking_compare/droidw_baseline

echo "Saved baseline output:"
find /content/vggt-demo/outputs_compare/droidw_baseline -maxdepth 3 -type f | grep -Ei "pose|traj|metric|timer|png|txt|csv" | head -50


## 6. Optional: rerun DROID-W + Ours and save separately

Use this only if you want to regenerate Ours from scratch. If you already saved Ours in Section 4, you can skip this section.

In [ ]:
%%bash
set -e

cd /content/vggt-demo/third_party/DROID-W

TORCH_LIB=$(python - <<'PY'
import torch, os
print(os.path.join(os.path.dirname(torch.__file__), "lib"))
PY
)

export LD_LIBRARY_PATH="$TORCH_LIB:$LD_LIBRARY_PATH"
export PYTHONPATH="/content/vggt-demo/third_party/DROID-W:$PYTHONPATH"

echo "==== Remove shared output folder to avoid mixing results ===="
rm -rf Outputs/Bonn/bonn_person_tracking

echo "==== Run DROID-W + Ours ===="
python run.py --config configs/Dynamic/Bonn/person_tracking_ours.yaml

echo "==== Save ours output ===="
mkdir -p /content/vggt-demo/outputs_compare
rm -rf /content/vggt-demo/outputs_compare/droidw_ours
cp -a Outputs/Bonn/bonn_person_tracking /content/vggt-demo/outputs_compare/droidw_ours

mkdir -p /content/drive/MyDrive/vggt_demo_results/bonn_person_tracking_compare
rm -rf /content/drive/MyDrive/vggt_demo_results/bonn_person_tracking_compare/droidw_ours
cp -a Outputs/Bonn/bonn_person_tracking \
      /content/drive/MyDrive/vggt_demo_results/bonn_person_tracking_compare/droidw_ours

echo "Saved ours output:"
find /content/vggt-demo/outputs_compare/droidw_ours -maxdepth 3 -type f | grep -Ei "pose|traj|metric|timer|png|txt|csv" | head -50


## 7. Run trajectory comparison metrics

In [ ]:
%cd /content/vggt-demo

!python scripts/eval/droidw_compare_eval.py \
  --baseline_dir /content/vggt-demo/outputs_compare/droidw_baseline \
  --ours_dir /content/vggt-demo/outputs_compare/droidw_ours \
  --output_dir /content/vggt-demo/outputs_compare/eval

print("\n==== Generated files ====")
!find /content/vggt-demo/outputs_compare/eval -maxdepth 1 -type f | sort


## 8. Show comparison tables and visualizations

In [ ]:
%cd /content/vggt-demo

import pandas as pd
from pathlib import Path
from IPython.display import display, Image

EVAL = Path("/content/vggt-demo/outputs_compare/eval")

print("==== Trajectory metrics ====")
display(pd.read_csv(EVAL / "trajectory_metrics_baseline_vs_ours.csv"))

print("==== Improvement, positive is better ====")
display(pd.read_csv(EVAL / "trajectory_improvement_baseline_vs_ours.csv"))

for p in [
    EVAL / "trajectory_xy_baseline_vs_ours.png",
    EVAL / "trajectory_xz_baseline_vs_ours.png",
    EVAL / "ate_error_over_frames.png",
    EVAL / "rpe_translation_error_over_frames.png",
    EVAL / "rpe_rotation_error_over_frames.png",
]:
    print("\n", p)
    if p.exists():
        display(Image(filename=str(p)))


## 9. Static-weight statistics

In [ ]:
%cd /content/vggt-demo

!python scripts/eval/static_weight_stats.py \
  --weight_dir /content/vggt-demo/outputs/droidw_ours_weights/person_tracking/static_weight_npy \
  --output_dir /content/vggt-demo/outputs_compare/eval/static_weight_stats \
  --dynamic_thresh 0.5

from IPython.display import Image, display
from pathlib import Path

STAT = Path("/content/vggt-demo/outputs_compare/eval/static_weight_stats")
for p in [
    STAT / "static_weight_stats_over_frames.png",
    STAT / "static_weight_temporal_change.png",
]:
    print("\n", p)
    if p.exists():
        display(Image(filename=str(p)))


## 10. Reconstruction metrics if ground-truth 3D is available

Run this only if you have a predicted point cloud and a ground-truth point cloud. Without GT geometry, true reconstruction quality cannot be measured reliably; use trajectory metrics and rendering/depth proxies instead.

In [ ]:
# Example only. Change paths before running.
%cd /content/vggt-demo

PRED_POINT_CLOUD = "/content/vggt-demo/path/to/pred_points.ply"
GT_POINT_CLOUD = "/content/vggt-demo/path/to/gt_points.ply"

# !python scripts/eval/reconstruction_metrics.py \
#   --pred "$PRED_POINT_CLOUD" \
#   --gt "$GT_POINT_CLOUD" \
#   --output_dir /content/vggt-demo/outputs_compare/eval/reconstruction \
#   --thresholds 0.01 0.02 0.05


## 11. Backup comparison results to Google Drive

In [ ]:
%%bash
set -e

cd /content/vggt-demo

SAVE_ROOT="/content/drive/MyDrive/vggt_demo_results/bonn_person_tracking_compare"
mkdir -p "$SAVE_ROOT"

rm -rf "$SAVE_ROOT/eval"
cp -a /content/vggt-demo/outputs_compare/eval "$SAVE_ROOT/eval"

cd "$SAVE_ROOT"
rm -f droidw_baseline.zip droidw_ours.zip eval.zip

zip -qr droidw_baseline.zip droidw_baseline
zip -qr droidw_ours.zip droidw_ours
zip -qr eval.zip eval

echo "Saved Drive backups:"
ls -lh "$SAVE_ROOT"/*.zip


## 12. Push metric scripts to GitHub main branch

Run this in VSCode, not necessarily in Colab:

```powershell
cd "D:\ESPACE\Foundation model\code\vggt_vscode_starter\vggt-demo"

git status
git checkout main
git pull origin main

git add scripts/eval/droidw_compare_eval.py `
        scripts/eval/reconstruction_metrics.py `
        scripts/eval/static_weight_stats.py `
        docs/droidw_eval_protocol.md `
        notebooks/VGGT_Any4D_DROIDW_Clean_Eval.ipynb

git commit -m "Add DROID-W evaluation metrics and clean Colab workflow"
git push origin main
```

If Git says there are uncommitted changes, either commit them first or save them with `git stash` before switching branches.
